In [ ]:
import numpy as np
from typing import Callable, Tuple, List

In [6]:
# 5.3

def w(x: np.float64) -> np.float64:
    return x**5 - x + 1

def dw(x : np.float64) -> np.float64:
    return 5 * x**4 - 1

ALPHA : float = -1.1673039782614186842560458998548

Result = Tuple[np.float64, int, int] # approx, iter, fun call

def bisection(a: np.float64, b: np.float64, tol: float, max_iter: int) -> Result:
    fa = w(a)
    n_feval = 1
    for i in range(1, max_iter + 1):
        if (b - a) / 2.0 < tol:
            return c, i, n_feval

        c = (a + b) / 2.0
        fc = w(c)
        n_feval += 1

        if fc == 0:
            return c, i, n_feval
        elif fa * fc < 0:
            b = c
        else:
            a, fa = c, fc

    return (a + b) / 2.0, max_iter, n_feval

def newton(x0: np.float64, tol: float, max_iter: int) -> Result:
    x_n = x0
    n_feval = 0
    for i in range(1, max_iter + 1):
        f_val = w(x_n)
        fp_val = dw(x_n)
        n_feval += 2


        if np.abs(fp_val) < 1e-16:
            return x_n, i, n_feval

        x_next = x_n - f_val / fp_val

        if np.abs(x_next - x_n) < tol:
            return x_next, i, n_feval
        x_n = x_next

    return x_n, max_iter, n_feval


def secant(x0: np.float64, x1: np.float64, tol: float, max_iter: int) -> Result:
    f0 = w(x0)
    f1 = w(x1)
    n_feval = 2

    for i in range(1, max_iter + 1):
        if np.abs(f1 - f0) < 1e-16:
            return x1, i, n_feval


        x_next = x1 - f1 * (x1 - x0) / (f1 - f0)

        if np.abs(x_next - x1) < tol:
            return x_next, i, n_feval

        x0, f0 = x1, f1
        x1, f1 = x_next, w(x_next)
        n_feval += 1

    return x1, max_iter, n_feval

def regula_falsi(a: np.float64, b: np.float64, tol: float, max_iter: int) -> Result:
    fa = w(a)
    fb = w(b)
    n_feval = 2
    c = a

    for i in range(1, max_iter + 1):

        if np.abs(b - a) < tol:
            return c, i, n_feval

        c = (fa * b - fb * a) / (fa - fb)
        fc = w(c)
        n_feval += 1

        if np.abs(fc) < 1e-16:
            return c, i, n_feval
        elif fa * fc < 0:
            b, fb = c, fc
        else:
            a, fa = c, fc

    return c, max_iter, n_feval

In [7]:
a, b = np.float64(-2.0), np.float64(-1.0)
x0_newton = np.float64(-1.2)
x0_sec, x1_sec = a, b

TOLERANCE = 1e-14
MAX_ITERATIONS = 100

print(f"Wielomian: w(x) = x^5 - x + 1")
print(f"Prawdziwy pierwiastek: alpha = {ALPHA}")
print(f"Tolerancja: {TOLERANCE}\n")


res_bisekcja = bisection(a, b, TOLERANCE, MAX_ITERATIONS)
res_newton = newton(x0_newton, TOLERANCE, MAX_ITERATIONS)
res_sieczne = secant(x0_sec, x1_sec, TOLERANCE, MAX_ITERATIONS)
res_regula = regula_falsi(a, b, TOLERANCE, MAX_ITERATIONS)


print("--- Wyniki ---")
print(f"{'Metoda':<15} {'Iteracje':<10} {'Obliczenia f':<15} {'Błąd końcowy':<20}")
print("-" * 60)

def print_result(name: str, result: Result):
    x_n, it, n_feval = result
    error = np.abs(x_n - ALPHA)
    print(f"{name:<15} {it:<10} {n_feval:<15} {error:<20.2e}")

print_result("Bisekcja", res_bisekcja)
print_result("Newton", res_newton)
print_result("Sieczne", res_sieczne)
print_result("Regula Falsi", res_regula)

Wielomian: w(x) = x^5 - x + 1
Prawdziwy pierwiastek: alpha = -1.1673039782614187
Tolerancja: 1e-14

--- Wyniki ---
Metoda          Iteracje   Obliczenia f    Błąd końcowy        
------------------------------------------------------------
Bisekcja        47         47              3.33e-15            
Newton          5          10              0.00e+00            
Sieczne         9          10              0.00e+00            
Regula Falsi    100        102             3.39e-13            


In [8]:
# 5.9



# --- 1. Funkcja testowa i jej dane ---
# f(x) = cos(x) - x
# Pierwiastek znaleziony z dużą dokładnością
ALPHA = 0.7390851332151606

def f(x: float) -> float:
    return np.cos(x) - x

def f_prime(x: float) -> float:
    return -np.sin(x) - 1

# --- 2. Implementacje metod (zwracające historię iteracji) ---

def newton_history(x0: float, max_iter: int = 10) -> List[float]:
    """Generuje listę przybliżeń x_n metodą Newtona."""
    iterates = [x0]
    x_n = x0
    for _ in range(max_iter):
        x_next = x_n - f(x_n) / f_prime(x_n)
        iterates.append(x_next)
        # Warunek stopu (osiągnięta precyzja)
        if np.abs(x_next - x_n) < 1e-15:
            break
        x_n = x_next
    return iterates

def secant_history(x0: float, x1: float, max_iter: int = 15) -> List[float]:
    """Generuje listę przybliżeń x_n metodą siecznych."""
    iterates = [x0, x1]
    f0, f1 = f(x0), f(x1)
    for _ in range(max_iter):
        # Unikamy dzielenia przez zero
        if np.abs(f1 - f0) < 1e-16:
            break

        x_next = x1 - f1 * (x1 - x0) / (f1 - f0)
        iterates.append(x_next)

        if np.abs(x_next - x1) < 1e-15:
            break

        x0, f0 = x1, f1
        x1, f1 = x_next, f(x_next)
    return iterates

# --- 3. Metoda L5.8: Estymator rzędu zbieżności ---

def estimate_order(iterates: List[float], alpha: float) -> np.ndarray:
    """
    Oblicza ciąg przybliżeń p_n na podstawie wzoru z L5.8.
    """
    # Krok a: Oblicz błędy
    errors = np.abs(np.array(iterates) - alpha)

    # Usuń błędy bliskie zeru maszynowemu, by uniknąć log(0)
    errors = errors[errors > 1e-16]

    # Krok b: Oblicz logarytmy błędów
    log_errors = np.log(errors)

    # Krok c: Oblicz różnice logarytmów (licznik i mianownik wzoru)
    # log_diffs[i] = log(e_{i+1}) - log(e_i) = log(e_{i+1} / e_i)
    log_diffs = np.diff(log_errors)

    # Krok d: Zastosuj wzór L5.8
    # p_n = (log(e_n+1) - log(e_n)) / (log(e_n) - log(e_n-1))
    # co odpowiada log_diffs[1:] / log_diffs[:-1]

    p_estimates = log_diffs[1:] / log_diffs[:-1]

    return p_estimates



if __name__ == "__main__":

    # Punkty startowe
    x_start_0 = 0.5
    x_start_1 = 1.0

    print("--- Zadanie L5.9: Eksperymentalne badanie rzędu zbieżności ---")

    # --- Test 1: Metoda Newtona (L5.6) ---
    print(f"\nTest Metody Newtona")
    newton_iterates = newton_history(x_start_0)
    p_estimates_newton = estimate_order(newton_iterates, ALPHA)

    print("Kolejne oszacowania p_n:")
    print(p_estimates_newton)
    print(f"Ostatnie oszacowanie: {p_estimates_newton[-1]:.6f}")

    # --- Test 2: Metoda Siecznych (L5.1) ---
    print(f"\nTest Metody Siecznych")
    secant_iterates = secant_history(x_start_0, x_start_1)
    p_estimates_secant = estimate_order(secant_iterates, ALPHA)

    print("Kolejne oszacowania p_n:")
    print(p_estimates_secant)
    print(f"Ostatnie oszacowanie: {p_estimates_secant[-1]:.6f}")

--- Zadanie L5.9: Eksperymentalne badanie rzędu zbieżności ---

Test Metody Newtona
Kolejne oszacowania p_n:
[ 2.0974481   1.99700948  1.38735172 -0.        ]
Ostatnie oszacowanie: -0.000000

Test Metody Siecznych
Kolejne oszacowania p_n:
[-33.80693893   1.01103532   1.94223534   1.51609994   1.68961863]
Ostatnie oszacowanie: 1.689619
